# total_GVA engineered features

Rebuild the 10 CAAFE-selected engineered features for 'log_total_GVA_2023' from the raw 'features.csv' columns, save the dataset (keys + features + target), then split the data by 'is_swindon' and fit a TabPFN model to predict GVA for Swindon.

In [13]:
import numpy as np
import pandas as pd
import warnings

warnings.filterwarnings("ignore")
features = pd.read_csv("../data preprocessing+EDA/features.csv", low_memory=False)
print("features", features.shape)

features (1129, 83)


In [14]:
L4 = "Level 4 qualifications or above: degree (BA, BSc), higher degree (MA, PhD, PGCE), NVQ level 4 to 5, HNC, HND, RSA Higher Diploma, BTEC Higher level, professional qualifications (for example, teaching, nursing, accountancy) %"

src_cols = [
    "VOA_total_RV_million_2023",
    "Working_Age_Pop",
    "LU_total_2025_msoa",
    "Total_Pop_Mid2024",
    L4,
    "No qualifications %",
    "LU_diversity_1-HHI_2025",
    "employment_rate_per_pop",
    "full_to_part_ratio",
    "share_enterprises_kibs_2025_msoa",
    "VOA_total_RV_pct_change",
]

ids = features[["LSOA21CD", "MSOA21CD"]].copy()
df = features[src_cols].copy()
print("selected source columns", df.shape)

selected source columns (1129, 11)


In [15]:
df["log_voa_rv_2023"] = np.log1p(df["VOA_total_RV_million_2023"])
df["rv_per_working_age"] = df["VOA_total_RV_million_2023"] / (df["Working_Age_Pop"] + 1)
df["sme_density"] = df["LU_total_2025_msoa"] / (df["Total_Pop_Mid2024"] / 1000)
df["qualification_index"] = df[L4] - df["No qualifications %"]
df["firm_size_diversity"] = df["LU_diversity_1-HHI_2025"]
df["employment_rate"] = df["employment_rate_per_pop"]
df["share_kibs"] = df["share_enterprises_kibs_2025_msoa"]
df["rv_pct_change"] = df["VOA_total_RV_pct_change"]

df["rv_per_employee"] = df["rv_per_working_age"] / (df["employment_rate"] + 1e-6)
df["sme_qual_interaction"] = df["sme_density"] * df["qualification_index"]
df["employment_quality"] = df["employment_rate"] * df["full_to_part_ratio"]
df["modern_sector_leverage"] = df["share_kibs"] * df["qualification_index"]
df["asset_growth_diversity"] = df["rv_pct_change"] * df["firm_size_diversity"]

In [16]:
final_features = [
    "log_voa_rv_2023",
    "rv_per_working_age",
    "sme_density",
    "qualification_index",
    "firm_size_diversity",
    "rv_per_employee",
    "sme_qual_interaction",
    "employment_quality",
    "modern_sector_leverage",
    "asset_growth_diversity",
]

X = df[final_features].copy()
X = X.fillna(X.median())
print("final feature set", X.shape)
X.head()

final feature set (1129, 10)


,log_voa_rv_2023,rv_per_working_age,sme_density,qualification_index,firm_size_diversity,rv_per_employee,sme_qual_interaction,employment_quality,modern_sector_leverage,asset_growth_diversity
0,11.833714,90.671356,127.416520,19.683656,0.896111,2579.526688,2508.022893,0.021090,9.021675,8.157646
1,16.268274,8704.430255,408.819476,13.321084,0.921664,1848.735672,5445.918610,1.324213,5.398086,2.748291
2,14.707645,1371.742970,170.408982,-0.120289,0.905896,1954.926910,-20.498273,0.116947,-0.034923,11.903836
3,13.536891,587.142758,234.936429,3.593145,0.905896,4248.534257,844.160745,0.138198,1.043171,14.288140
4,14.753778,2255.558815,205.900430,11.370621,0.904844,8155.068718,2341.215710,0.138291,4.625337,4.415880


In [17]:
target = "log_total_GVA_2023"
upd = pd.concat([pd.read_csv("../data preprocessing+EDA/train_updated.csv"),
                 pd.read_csv("../data preprocessing+EDA/test_updated.csv")], ignore_index=True)

dataset = pd.concat([ids, X], axis=1).merge(upd[["LSOA21CD", target, "is_swindon"]], on="LSOA21CD", how="left")
dataset = dataset.dropna(subset=[target, "MSOA21CD"]).reset_index(drop=True)
dataset.to_csv("total_gva_engineered_features.csv", index=False, encoding="utf-8")
print("saved total_gva_engineered_features.csv", dataset.shape)

saved total_gva_engineered_features.csv (1125, 14)


## TabPFN model

In [18]:
from tabpfn import TabPFNRegressor
from tabpfn.constants import ModelVersion
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score

data = dataset.dropna(subset=[target, "MSOA21CD"]).reset_index(drop=True)
X_mat = data[final_features].values
y = data[target].values
groups = data["MSOA21CD"].values
print("modelling rows", X_mat.shape)

modelling rows (1125, 10)


In [19]:
gkf = GroupKFold(n_splits=5)
pred = np.empty(len(y))
for tr, va in gkf.split(X_mat, y, groups):
    reg = TabPFNRegressor.create_default_for_version(ModelVersion.V3)
    reg.fit(X_mat[tr], y[tr])
    pred[va] = reg.predict(X_mat[va])

print(f"target: {target} (GroupKFold OOF)")
print(f"MAE = {mean_absolute_error(y, pred):.4f}")
print(f"RMSE = {np.sqrt(mean_squared_error(y, pred)):.4f}")
print(f"MAPE = {mean_absolute_percentage_error(y, pred):.4f}")
print(f"R2 = {r2_score(y, pred):.4f}")

target: log_total_GVA_2023 (GroupKFold OOF)
MAE = 0.3875
RMSE = 0.6348
MAPE = 0.1056
R2 = 0.6798


## Split by is_swindon

Train on the rest of the country 'Others' and test on 'Swindon' only.

In [20]:
train = data[data["is_swindon"] == "Others"].reset_index(drop=True)
test  = data[data["is_swindon"] == "Swindon"].reset_index(drop=True)

X_tr, X_te = train[final_features].values, test[final_features].values
y_tr, y_te = train[target].values, test[target].values
print("train(Others)", X_tr.shape, "test(Swindon)", X_te.shape)

train(Others) (988, 10) test(Swindon) (137, 10)


In [21]:
# Save the Swindon-only rows
swindon = data[data["is_swindon"] == "Swindon"].reset_index(drop=True)
swindon_cols = ["LSOA21CD", "MSOA21CD"] + final_features + [target, "is_swindon"]
swindon[swindon_cols].to_csv("data_swindon.csv", index=False, encoding="utf-8")
print("saved swindon data", swindon[swindon_cols].shape)

saved swindon data (137, 14)


In [22]:
# Swindon result
reg = TabPFNRegressor.create_default_for_version(ModelVersion.V3)
reg.fit(X_tr, y_tr)
pred = reg.predict(X_te)

print(f"target: {target} (train=Others, test=Swindon)")
print(f"MAE = {mean_absolute_error(y_te, pred):.4f}")
print(f"RMSE = {np.sqrt(mean_squared_error(y_te, pred)):.4f}")
print(f"MAPE = {mean_absolute_percentage_error(y_te, pred):.4f}")
print(f"R2 = {r2_score(y_te, pred):.4f}")

target: log_total_GVA_2023 (train=Others, test=Swindon)
MAE = 0.4469
RMSE = 0.7137
MAPE = 0.1237
R2 = 0.6935


## XGBoost model

Same 10 features and the same two evaluations (GroupKFold OOF + is_swindon split), using XGBoost for comparison and for SHAP-based attribution downstream.

In [23]:
from xgboost import XGBRegressor

gkf = GroupKFold(n_splits=5)
pred = np.empty(len(y))
for tr, va in gkf.split(X_mat, y, groups):
    xgb = XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=4,
                       subsample=0.8, colsample_bytree=0.8, random_state=42)
    xgb.fit(X_mat[tr], y[tr])
    pred[va] = xgb.predict(X_mat[va])

print(f"target: {target} (XGBoost GroupKFold OOF)")
print(f"MAE = {mean_absolute_error(y, pred):.4f}")
print(f"RMSE = {np.sqrt(mean_squared_error(y, pred)):.4f}")
print(f"MAPE = {mean_absolute_percentage_error(y, pred):.4f}")
print(f"R2 = {r2_score(y, pred):.4f}")

target: log_total_GVA_2023 (XGBoost GroupKFold OOF)
MAE = 0.4792
RMSE = 0.7074
MAPE = 0.1331
R2 = 0.6024


In [24]:
# Swindon
xgb = XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=4,
                   subsample=0.8, colsample_bytree=0.8, random_state=42)
xgb.fit(X_tr, y_tr)
pred = xgb.predict(X_te)

print(f"target: {target} (XGBoost train=Others, test=Swindon)")
print(f"MAE = {mean_absolute_error(y_te, pred):.4f}")
print(f"RMSE = {np.sqrt(mean_squared_error(y_te, pred)):.4f}")
print(f"MAPE = {mean_absolute_percentage_error(y_te, pred):.4f}")
print(f"R2 = {r2_score(y_te, pred):.4f}")

target: log_total_GVA_2023 (XGBoost train=Others, test=Swindon)
MAE = 0.5006
RMSE = 0.7215
MAPE = 0.1396
R2 = 0.6868


## Uncertainty analysis — paired bootstrap on the fixed test split

In [ ]:
reg_llm = TabPFNRegressor.create_default_for_version(ModelVersion.V3)
reg_llm.fit(X_tr, y_tr)
pred_tabpfn_llm = reg_llm.predict(X_te)

xgb_llm = XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=4,
                       subsample=0.8, colsample_bytree=0.8, random_state=42)
xgb_llm.fit(X_tr, y_tr)
pred_xgb_llm = xgb_llm.predict(X_te)

print("TabPFN  (LLM feats) MAE =", mean_absolute_error(y_te, pred_tabpfn_llm),
      " RMSE =", np.sqrt(mean_squared_error(y_te, pred_tabpfn_llm)))
print("XGBoost (LLM feats) MAE =", mean_absolute_error(y_te, pred_xgb_llm),
      " RMSE =", np.sqrt(mean_squared_error(y_te, pred_xgb_llm)))

In [ ]:
MANUAL_FEATURES = [
    "Urban_rura_Urban",
    "LU_pct_micro_2025_msoa",
    "LU_pct_large_2025_msoa",
    "log_enterprises_per_1k_residents_2025",
    "turnover_diversity_1-HHI_2025_msoa",
    "LU_diversity_1-HHI_2025",
    "share_enterprises_kibs_2025_msoa",
    "emp_rate",
    "full_time_share",
    "log_Mid-2024 population",
    "IMD_decile",
]

manual_full = data[["LSOA21CD", "is_swindon"]].merge(
    upd[["LSOA21CD"] + MANUAL_FEATURES], on="LSOA21CD", how="left"
)
manual_full[MANUAL_FEATURES] = manual_full[MANUAL_FEATURES].fillna(manual_full[MANUAL_FEATURES].median())

mask_train_manual = manual_full["is_swindon"] == "Others"
mask_test_manual = manual_full["is_swindon"] == "Swindon"

X_tr_manual = manual_full.loc[mask_train_manual, MANUAL_FEATURES].values
X_te_manual = manual_full.loc[mask_test_manual, MANUAL_FEATURES].values

# Sanity check: same row count and same LSOA ordering as the LLM-feature test split.
assert X_te_manual.shape[0] == X_te.shape[0] == len(y_te)
assert (manual_full.loc[mask_test_manual, "LSOA21CD"].values == test["LSOA21CD"].values).all()

reg_manual = TabPFNRegressor.create_default_for_version(ModelVersion.V3)
reg_manual.fit(X_tr_manual, y_tr)
pred_tabpfn_manual = reg_manual.predict(X_te_manual)

xgb_manual = XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=4,
                          subsample=0.8, colsample_bytree=0.8, random_state=42)
xgb_manual.fit(X_tr_manual, y_tr)
pred_xgb_manual = xgb_manual.predict(X_te_manual)

print("TabPFN  (manual feats) MAE =", mean_absolute_error(y_te, pred_tabpfn_manual),
      " RMSE =", np.sqrt(mean_squared_error(y_te, pred_tabpfn_manual)))
print("XGBoost (manual feats) MAE =", mean_absolute_error(y_te, pred_xgb_manual),
      " RMSE =", np.sqrt(mean_squared_error(y_te, pred_xgb_manual)))

In [ ]:
from scipy.stats import wilcoxon

def _mae(y, p):
    return np.mean(np.abs(y - p))

def _rmse(y, p):
    return np.sqrt(np.mean((y - p) ** 2))

def paired_bootstrap(y_true, pred_a, pred_b, n_boot=10000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    mae_diffs = np.empty(n_boot)
    rmse_diffs = np.empty(n_boot)

    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        yb, ab, bb = y_true[idx], pred_a[idx], pred_b[idx]
        mae_diffs[b] = _mae(yb, ab) - _mae(yb, bb)
        rmse_diffs[b] = _rmse(yb, ab) - _rmse(yb, bb)

    mae_point = _mae(y_true, pred_a) - _mae(y_true, pred_b)
    rmse_point = _rmse(y_true, pred_a) - _rmse(y_true, pred_b)

    ae_a = np.abs(y_true - pred_a)
    ae_b = np.abs(y_true - pred_b)
    try:
        wstat, wp = wilcoxon(ae_a, ae_b)
    except ValueError:
        wstat, wp = np.nan, np.nan

    return {
        "n_test": n,
        "n_boot": n_boot,
        "MAE_diff": mae_point,
        "MAE_ci95": tuple(np.percentile(mae_diffs, [2.5, 97.5])),
        "RMSE_diff": rmse_point,
        "RMSE_ci95": tuple(np.percentile(rmse_diffs, [2.5, 97.5])),
        "wilcoxon_stat": wstat,
        "wilcoxon_p": wp,
    }

def report(title, result):
    lo_m, hi_m = result["MAE_ci95"]
    lo_r, hi_r = result["RMSE_ci95"]
    print(f"\n=== {title} (n={result['n_test']}, {result['n_boot']} bootstrap reps) ===")
    print(f"  MAE  diff (A-B) = {result['MAE_diff']:+.4f}   95% CI [{lo_m:+.4f}, {hi_m:+.4f}]")
    print(f"  RMSE diff (A-B) = {result['RMSE_diff']:+.4f}   95% CI [{lo_r:+.4f}, {hi_r:+.4f}]")
    print(f"  Wilcoxon signed-rank on |error|: stat={result['wilcoxon_stat']:.2f}, p={result['wilcoxon_p']:.4f}")

In [ ]:
# A: TabPFN vs XGBoost, both on the LLM (CAAFE) feature set
res_model = paired_bootstrap(y_te, pred_tabpfn_llm, pred_xgb_llm, n_boot=10000, seed=42)
report("TabPFN vs XGBoost (LLM/CAAFE features)", res_model)

# B: LLM-selected vs manually-selected features, primary model = TabPFN
res_feat_tabpfn = paired_bootstrap(y_te, pred_tabpfn_llm, pred_tabpfn_manual, n_boot=10000, seed=42)
report("LLM-selected vs manually-selected features (TabPFN)", res_feat_tabpfn)

# B (secondary check): same feature comparison, scored with XGBoost
res_feat_xgb = paired_bootstrap(y_te, pred_xgb_llm, pred_xgb_manual, n_boot=10000, seed=42)
report("LLM-selected vs manually-selected features (XGBoost, secondary check)", res_feat_xgb)

summary = pd.DataFrame([
    {"comparison": "TabPFN vs XGBoost (LLM feats)", **{k: v for k, v in res_model.items() if k not in ("n_test", "n_boot")}},
    {"comparison": "LLM vs manual feats (TabPFN)", **{k: v for k, v in res_feat_tabpfn.items() if k not in ("n_test", "n_boot")}},
    {"comparison": "LLM vs manual feats (XGBoost)", **{k: v for k, v in res_feat_xgb.items() if k not in ("n_test", "n_boot")}},
])
summary.to_csv("bootstrap_uncertainty_summary.csv", index=False)
summary